# Automatización de Smartsheet: Asignación de códigos y descarga de archivos adjuntos / Smartsheet Automation: Code allocation & Attachments Download

## 0. Preparación / Preparation

**ES:** Cómo obtener el ID de Smartsheet y el token de acceso:

1. **ID de Smartsheet:** Lo primero que debe hacer es obtener el ID de la hoja de Smartsheet. Puede encontrarlo en la URL de la hoja de Smartsheet o en las propiedades de la hoja.
2. **Token de acceso:** Debe generar un token de acceso en Smartsheet. Vaya a su perfil de Smartsheet → Aplicaciones y integraciones → Acceso a la API → Generar nuevo token de acceso.
3. Guarde ambos valores en su archivo `.env`.

---

**EN:** How to get the Smartsheet ID and Smartsheet access token:

1. **Smartsheet ID:** The first thing you must have is the Smartsheet sheet ID. You can find it in the Smartsheet sheet URL or in the sheet properties.
2. **Access token:** You need to generate an access token in Smartsheet. Go to your Smartsheet profile → Apps & Integrations → API Access → Generate new access token.
3. Save both values in your `.env` file.

## 1. Configuración del entorno / Environment setting

In [ ]:
# Install dotenv
# %pip install python-dotenv
# %pip install rarfile
# %pip install smartsheet-python-sdk

In [ ]:
import smartsheet
import requests
import os
import zipfile
import arcpy
import atexit
import shutil
from dotenv import load_dotenv

global sheet_id_c1
global sheet_id_c2
global sheet_id_c3
global smartsheet_client

# NEED TO SET UP USERS SMARTSHEET TOKEN in your ENV file
# https://smartsheet.redoc.ly/
from pathlib import Path
load_dotenv(Path(os.getcwd()).parent / '.env')

SMARTSHEET_ACCESS_TOKEN = os.environ.get('TOKEN')
smartsheet_client = smartsheet.Smartsheet(SMARTSHEET_ACCESS_TOKEN)

#Smartsheet ID setting by the AR Project
sheet_id_c1 = os.environ.get('SHEET_C1')
sheet_id_c2 = os.environ.get('SHEET_C2')
sheet_id_c3 = os.environ.get('SHEET_C3')

# Set the path to the Windows folder where you want to save the attachments
path_to_folder1 = os.environ.get('FOLDER_C1')
path_to_folder2 = os.environ.get('FOLDER_C2')
path_to_folder3 = os.environ.get('FOLDER_C3')

## 2. Funciones / Functions

### 2.1. ACTUALIZAR CÓDIGO DE LA ACTIVIDAD usando FECHA, COMPONENTE y persona que reporta en Smartsheet / UPDATE Activity Code using DATE, COMPONENT, and person who reports to Smartsheet

In [ ]:
#Class Version - updated 2024-06-27
import itertools


class SmartsheetOperations:
    def __init__(self):
        self.existing_cdg_values = set()
        
    def get_ssheet_AR(self, sheet_id):
        return smartsheet_client.Sheets.get_sheet(sheet_id)

    def existing_cdgs(self, sheet_id):
        ssheet = self.get_ssheet_AR(sheet_id)
        column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")

        for row in ssheet.rows:
            codigo_cell = row.get_column(column_CdgActvdd.id_)
            if codigo_cell.value:
                self.existing_cdg_values.add(codigo_cell.value)

        return self.existing_cdg_values

class CDGGenerator:
    def __init__(self, existing_cdg_values):
        self.existing_cdg_values = existing_cdg_values
        
    
    def abc_order(self, date_srt, text):
        # Generate the initial part of the code
        cdg_base = f"{date_srt}_{text}_"

        # Generate combinations of letters
        for length in range(1, 5):  # Adjust the range as needed for more combinations
            for combo in itertools.product("abcdefghijklmnopqrstuvwxyz", repeat=length):
                suffix = ''.join(combo)
                new_cdg = cdg_base + suffix
                if new_cdg not in self.existing_cdg_values:
                    self.existing_cdg_values.add(new_cdg)
                    return new_cdg

        return None

class Updater:
    def __init__(self):
        self.smartsheet_operations = SmartsheetOperations()
        self.sheet_id_c1 = sheet_id_c1
        self.sheet_id_c2 = sheet_id_c2
        self.sheet_id_c3 = sheet_id_c3

    def updateCdgActvdd(self, sheet_id, begin_rowNum, EndRowNum=2022):
        compo = self.get_component(sheet_id)
        ssheet = self.smartsheet_operations.get_ssheet_AR(sheet_id)
        row_count = len(ssheet.rows)
        if EndRowNum == 2022:
            EndRowNum = row_count

        columns = {
            "FECHA": ssheet.get_column_by_title("FECHA DE LA ACTIVIDAD"),
            "CdgActvdd": ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD"),
            "QUIEN": ssheet.get_column_by_title("NOMBRE DE QUIEN REPORTA"),
            "cal_dato": ssheet.get_column_by_title("CALIDAD DE DATOS"),
            "cal_sig": ssheet.get_column_by_title("Calidad SIG")
        }
        if compo == "C2":
            columns["ORG"] = ssheet.get_column_by_title("ORGANIZACIÓN")

        existing_cdg_values = self.smartsheet_operations.existing_cdgs(sheet_id)
        cdg_generator = CDGGenerator(existing_cdg_values)
        review_statuses = {"Rojo", "Amarillo", "Verde", "Sí", "Espera", "No"}

        for row in ssheet.rows[begin_rowNum:EndRowNum]:
            if not row.modified_at:
                continue

            val_dato = row.get_column(columns["cal_dato"].id_).value
            val_sig = row.get_column(columns["cal_sig"].id_).value if compo != "C3" else "Null"

            if val_dato in review_statuses or val_sig in review_statuses:
                print(f"Row {row.row_number} already assigned the code - Bnf: {val_dato}, Area: {val_sig}")
                continue

            fecha_cell = row.get_column(columns["FECHA"].id_)
            if not fecha_cell.value:
                continue

            fecha = fecha_cell.value.replace("-", "")
            if compo == "C2":
                dequien = row.get_column(columns["ORG"].id_).value
            else:
                dequien = str(row.get_column(columns["QUIEN"].id_).value[:3])

            if not dequien:
                print("No hay datos de quien u organizaion")
                continue

            text = f"{compo}_{dequien}"
            conca_values = cdg_generator.abc_order(str(fecha[2:]), text)

            updated_cell = smartsheet.models.Cell({
                "column_id": columns["CdgActvdd"].id_,
                "value": conca_values
            })
            updated_row = smartsheet.models.Row({
                "id": row.id,
                "cells": [updated_cell]
            })

            try:
                result = smartsheet_client.Sheets.update_rows(sheet_id, [updated_row]).data
                print(f"Updated Row {row.row_number}: {updated_row.cells[0].value}")
            except smartsheet.exceptions.SmartsheetAPIError as e:
                print(f"An error occurred: {e}")

    def get_component(self, sheet_id):
        if sheet_id == self.sheet_id_c1:
            return "C1"
        elif sheet_id == self.sheet_id_c2:
            return "C2"
        elif sheet_id == self.sheet_id_c3:
            return "C3"
        else:
            raise ValueError("Invalid sheet_id")

    def update_columns(self, sheet_id, column_name, new_cell_value, begin_rowNum, EndRowNum=2022):
        ssheet = self.smartsheet_operations.get_ssheet_AR(sheet_id)
        row_count = len(ssheet.rows)
        if EndRowNum == 2022:
            EndRowNum = row_count

        column = ssheet.get_column_by_title(column_name)
        column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")

        for row in ssheet.rows[begin_rowNum:EndRowNum]:
            if not row.modified_at:
                continue

            codigo_cell = row.get_column(column_CdgActvdd.id_)
            if not codigo_cell.value:
                continue

            updated_cell = smartsheet.models.Cell({
                "column_id": column.id_,
                "value": new_cell_value
            })
            updated_row = smartsheet.models.Row({
                "id": row.id,
                "cells": [updated_cell]
            })

            try:
                result = smartsheet_client.Sheets.update_rows(sheet_id, [updated_row]).data
                print(f"Updated Row {row.row_number}: {updated_row.cells[0].value}")
            except smartsheet.exceptions.SmartsheetAPIError as e:
                print(f"An error occurred: {e}")
    
    def update_review_status(self, sheet_id, begin_rowNum, EndRowNum=2022):
        """
        Update review statuses based on data values:
        - If 'ACCIONES DE RESTAURACIÓN AbE' or 'TOTAL DE HECTÁREAS' has values: 'Espera'
        - If no values: 'Sí'
        Only updates Calidad SIG status.
        """
        compo = self.get_component(sheet_id)
        ssheet = self.smartsheet_operations.get_ssheet_AR(sheet_id)
        row_count = len(ssheet.rows)
        if EndRowNum == 2022:
            EndRowNum = row_count

        # Get column IDs
        columns = {
            "CdgActvdd": ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD"),
            "cal_sig": ssheet.get_column_by_title("Calidad SIG"),
            "ABE": ssheet.get_column_by_title("ACCIONES DE RESTAURACIÓN AbE"),
            "HA": ssheet.get_column_by_title("TOTAL DE HECTÁREAS")
        }

        review_statuses = {"Rojo", "Amarillo", "Verde", "Sí", "Espera", "No"}
        
        for row in ssheet.rows[begin_rowNum:EndRowNum]:
            if not row.modified_at:
                continue
                
            # Check if row has codigo
            codigo_cell = row.get_column(columns["CdgActvdd"].id_)
            if not codigo_cell.value:
                print(f"Row {row.row_number} doesn't have a code, skipping.")
                continue
                
            # Skip C3 component as it doesn't have Calidad SIG
            if compo == "C3":
                print(f"Row {row.row_number} is from C3, skipping SIG update.")
                continue
                
            # Get current Calidad SIG value
            val_sig = row.get_column(columns["cal_sig"].id_).value
            
            # Skip if already reviewed
            if val_sig in review_statuses:
                print(f"Row {row.row_number} already reviewed - Area: {val_sig}")
                continue
                
            # Get values from the AbE and HA columns
            abe_value = row.get_column(columns["ABE"].id_).value
            ha_value = row.get_column(columns["HA"].id_).value
            
            # Prepare cell to update
            updated_cells = []
            
            # Set Calidad SIG status based on both ABE and HA values
            if abe_value or ha_value:
                cal_sig_value = "Espera"  # If either ABE or HA value exists, set to "Espera"
            else:
                cal_sig_value = "Sí"      # If no values, set to "Sí"
                
            updated_cells.append(smartsheet.models.Cell({
                "column_id": columns["cal_sig"].id_,
                "value": cal_sig_value
            }))
            
            # Update the row with new values
            updated_row = smartsheet.models.Row({
                "id": row.id,
                "cells": updated_cells
            })

            try:
                result = smartsheet_client.Sheets.update_rows(sheet_id, [updated_row]).data
                print(f"Updated Row {row.row_number}: Cal_sig={cal_sig_value}")
            except smartsheet.exceptions.SmartsheetAPIError as e:
                print(f"An error occurred: {e}")



### 2.2. OBTENER archivos adjuntos de Smartsheet / GET Attachments from Smartsheet

#### 2.2.1. OBTENER archivos adjuntos de Smartsheet: Clase / GET Attachments from Smartsheet: Class

In [ ]:
import os
import re
import requests
import zipfile
#import rarfile
import arcpy
import shutil

# Global/session-level attachment guard shared by all FileHandler instances
PROCESSED_ATTACHMENT_IDS = set()

class SmartsheetHandler:
    def __init__(self, sheet_id):
        self.sheet_id = sheet_id

    def get_ssheet_AR(self):
        ssheet = smartsheet_client.Sheets.get_sheet(self.sheet_id)
        return ssheet

    def get_cdg_value(self, row_number):
        ssheet = self.get_ssheet_AR()
        rows = ssheet.rows
        column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")
        for row in rows[row_number-1:row_number]:
            if row.get_column(column_CdgActvdd.id_).value:
                cdg_value = row.get_column(column_CdgActvdd.id_).value
            else:
                print(f"row: {row.row_number}, No hay codigo")
                return None
        return cdg_value

    def get_cdg_area(self, row_start, row_end):
        ssheet = self.get_ssheet_AR()
        rows = ssheet.rows
        cdg_value = []
        area_value = []
        column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")
        column_HA = ssheet.get_column_by_title("TOTAL DE HECTÁREAS")
        for row in rows[row_start:row_end]:
            if row.get_column(column_CdgActvdd.id_).value and row.get_column(column_HA.id_).value:
                cdg_value.append(row.get_column(column_CdgActvdd.id_).value)
                area_value.append(row.get_column(column_HA.id_).value)
            else:
                print(f"row: {row.row_number}, No hay datos de Area")
        return cdg_value, area_value

    def create_c2_cdg_area_dict(self, row_start, row_end):
        cdg_values, area_values = self.get_cdg_area(row_start, row_end)
        c2_cdg_area_dict = {cdg: round(area, 2) for cdg, area in zip(cdg_values, area_values)}
        return c2_cdg_area_dict

    def get_org_f_name(self, row_number):
        ssheet = self.get_ssheet_AR()
        rows = ssheet.rows
        column_org_code = ssheet.get_column_by_title("NÚMERO DE CONTRATO")
        column_org_name = ssheet.get_column_by_title("ORGANIZACIÓN")
        column_org_trimes = ssheet.get_column_by_title("TRIMESTRE QUE REPORTA")
        for row in rows[row_number-1:row_number]:
            if row.get_column(column_org_code.id_).value and row.get_column(column_org_trimes.id_).value:
                org_cdg = row.get_column(column_org_code.id_).value
                org_name = row.get_column(column_org_name.id_).value
                org_trimes = row.get_column(column_org_trimes.id_).value
                org_f_name = f"{org_cdg}_{org_name}_{org_trimes}"
                return org_f_name
            else:
                print(f"row: {row.row_number}, No hay datos de Org or de Trimestre")
                return None
        return None

    def process_attachments(self, file_handler, arcgis_handler, begin_rowNum, EndRowNum=None):
        if EndRowNum is None:
            EndRowNum = 2022

        ssheet = self.get_ssheet_AR()
        if ssheet is None:
            return
        rows = ssheet.rows
        row_count = len(rows)
        if EndRowNum == 2022:
            EndRowNum = row_count

        compo = None
        if self.sheet_id == sheet_id_c1:
            compo = "C1"
        elif self.sheet_id == sheet_id_c2:
            compo = "C2"
        elif self.sheet_id == sheet_id_c3:
            compo = "C3"
        else:
            print("Smartsheet 'id' doesn't match as of the project AR")
            return

        if compo == 'C2':
            c2_cdg_area_dict = self.create_c2_cdg_area_dict(begin_rowNum, EndRowNum)

        for actv in rows[begin_rowNum:EndRowNum]:
            row = smartsheet_client.Sheets.get_row(
                self.sheet_id, actv.id_, include='discussions,attachments,columns,columnType'
            )

            print(f"Checking row {row.row_number}")

            for attachment in row.attachments:
                if compo == 'C2':
                    cdg_value = self.get_org_f_name(row.row_number)
                else:
                    cdg_value = self.get_cdg_value(row.row_number)

                if cdg_value is not None:
                    file_type, shape_extract_folder = file_handler.save_attachment(self.sheet_id, attachment, cdg_value)

                    if file_type == 'Shapes':
                        if compo == "C1":
                            arcgis_handler.shp_add_to_arcgispro(shape_extract_folder, cdg_value)
                        elif compo == "C2":
                            arcgis_handler.shp_add_to_arcgispro_c2(shape_extract_folder, cdg_value, c2_cdg_area_dict)

class FileHandler:
    @staticmethod
    def create_directory(path, dir_name):
        dir_path = os.path.join(path, dir_name)
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    @staticmethod
    def extract_zip(file_path, destination_path):
        extracted_files = []
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(destination_path)
            extracted_files = zip_ref.namelist()
        return extracted_files

    @staticmethod
    def extract_rar(file_path, destination_path):
        extracted_files = []
        try:
            import rarfile  # lazy import so users without rarfile can still run other flows
        except ImportError:
            print("rarfile package not installed. Skipping RAR extraction.")
            return extracted_files
        with rarfile.RarFile(file_path) as rar_ref:
            rar_ref.extractall(destination_path)
            extracted_files = rar_ref.namelist()
        return extracted_files

    @staticmethod
    def find_shapefiles(folder):
        shapefiles = []
        for root, _, files in os.walk(folder):
            for file in files:
                if file.lower().endswith('.shp'):
                    shapefiles.append(os.path.join(root, file))
        return shapefiles

    @staticmethod
    def _sanitize_shapefile_basename(name: str) -> str:
        # ArcGIS shapefile names cannot safely contain additional '.' characters before the extension
        # Replace dots and any unusual characters with underscores, keep dashes/underscores
        sanitized = name.replace('.', '_')
        sanitized = re.sub(r'[^A-Za-z0-9_\-]', '_', sanitized)
        return sanitized

    @staticmethod
    def rename_and_move_files(shapefile, shape_extract_folder, base_name):
        # Get the directory and original filename without extension
        shapefile_dir = os.path.dirname(shapefile)
        original_name = os.path.splitext(os.path.basename(shapefile))[0]
        
        # Sanitize base name for ArcGIS (do not rename the folder, only the files)
        new_base_name = FileHandler._sanitize_shapefile_basename(base_name)
        if new_base_name != base_name:
            print(f"Note: sanitized shapefile base name from '{base_name}' to '{new_base_name}' for ArcGIS compatibility")
        
        # List of all possible shapefile-related extensions
        all_extensions = ['.shp', '.dbf', '.prj', '.shx', '.cpg', '.qmd', '.sbn', '.sbx', '.fbn', '.fbx', '.ain', '.aih', '.atx', '.ixs', '.mxs', '.shp.xml']
        
        for ext in all_extensions:
            if ext == '.shp.xml':
                # Handle compound extension
                old_file = os.path.join(shapefile_dir, original_name + '.shp.xml')
            else:
                old_file = os.path.join(shapefile_dir, original_name + ext)
            
            # Only process if the old file exists
            if os.path.exists(old_file):
                if ext == '.shp.xml':
                    new_file = os.path.join(shape_extract_folder, new_base_name + '.shp.xml')
                else:
                    new_file = os.path.join(shape_extract_folder, new_base_name + ext)
                
                # Handle duplicates (though this shouldn't happen with this approach)
                i = 1
                while os.path.exists(new_file):
                    if ext == '.shp.xml':
                        new_file = os.path.join(shape_extract_folder, f"{new_base_name}_{i}.shp.xml")
                    else:
                        new_file = os.path.join(shape_extract_folder, f"{new_base_name}_{i}{ext}")
                    i += 1
                
                try:
                    shutil.move(old_file, new_file)
                    print(f"Moved and renamed file {os.path.basename(old_file)} to {os.path.basename(new_file)}")
                except Exception as e:
                    print(f"Error moving file {old_file}: {e}")

    def save_attachment(self, sheet_id, attachment, cdg_value):
        # Duplicate-guard: skip same attachment id across the entire session
        try:
            att_id = attachment.id_
        except Exception:
            att_id = None
        if att_id and att_id in PROCESSED_ATTACHMENT_IDS:
            print(f"Skipping already-processed attachment id {att_id} ({attachment.name})")
            return None, None

        data = smartsheet_client.Attachments.get_attachment(sheet_id, attachment.id_)
        print(f"Checking attachment {data.name}")

        if data.url is None:
            print(f"Skipping attachment {data.name} because URL is None")
            return None, None

        f_name = os.path.splitext(data.name)[0].lower()
        f_exten_name = os.path.splitext(data.name)[1].lower()
        if f_exten_name in ['.xlsm']:
            extension = f_exten_name
            file_type = 'Reporte-pp'
        elif f_exten_name in ['.xls', '.xlsx']:
            extension = f_exten_name
            if any(keyword in f_name for keyword in ['bd', 'bene', 'm&e', 'basedatos']):
                file_type = 'Reporte-pp'
            elif any(keyword in f_name for keyword in ['Área', 'área', 'area', 'hectareas', 'ha', 'SAF', 'saf', '_reas_']):
                file_type = 'Area_ha'
            else:
                file_type = 'Referencia'
        elif f_exten_name in ['.doc', '.docx', '.pdf', '.pptx', '.ppt']:
            extension = f_exten_name
            file_type = 'Verficador'
        elif f_exten_name in ['.jpg', '.jpeg', '.png', '.jfif']:
            extension = f_exten_name
            file_type = 'Fotos'
        elif f_exten_name in ['.zip', '.7z', '.rar']:
            extension = f_exten_name
            if any(keyword in f_name for keyword in ['anexos']):
                return 'Anexos', None  # Skip Anexos attachments
            if any(keyword in f_name for keyword in ['shape', 'shapes', 'shp', 'ha', 'saf', 'prote', 'plant', 'resta', 'produ']):
                file_type = 'Shapes'
            else:
                file_type = 'Anexos'
        else:
            extension = ''
            file_type = ''

        if sheet_id == sheet_id_c1:
            path_to_folder = path_to_folder1
        elif sheet_id == sheet_id_c2:
            path_to_folder = path_to_folder2
        elif sheet_id == sheet_id_c3:
            path_to_folder = path_to_folder3
        else:
            print("Smartsheet 'id' doesn't match as of the project AR")
            return None, None

        dir_name = str(cdg_value)
        self.create_directory(path_to_folder, dir_name)

        abs_path_to_folder = os.path.abspath(path_to_folder)
        file_path = os.path.join(abs_path_to_folder, dir_name, f"{cdg_value}-{file_type}{extension}")

        if os.path.isfile(file_path):
            i = 1
            while os.path.isfile(file_path):
                i += 1
                file_path = os.path.join(abs_path_to_folder, dir_name, f"{cdg_value}-{file_type}-{i}{extension}")

        if 'Anexos' in file_path:
            return file_type, None

        attachment_data = requests.get(data.url).content

        with open(file_path, 'wb') as f:
            f.write(attachment_data)

        print(f"Downloaded attachment {data.name} to {file_path}")

        # Mark this attachment as processed globally after a successful download
        if att_id:
            PROCESSED_ATTACHMENT_IDS.add(att_id)

        if 'Shapes' in file_path:
            shape_zip_path = file_path
            # Use the zip file name (without extension) as the extraction folder
            shape_extract_folder = os.path.splitext(file_path)[0]
            if not os.path.exists(shape_extract_folder):
                os.makedirs(shape_extract_folder, exist_ok=True)
            
            if f_exten_name == '.zip':
                extracted_files = self.extract_zip(shape_zip_path, shape_extract_folder)
                print(f"Extracted shapes zip file {data.name} to {shape_extract_folder}")
            elif f_exten_name == '.rar':
                extracted_files = self.extract_rar(shape_zip_path, shape_extract_folder)
                print(f"Extracted shapes rar file {data.name} to {shape_extract_folder}")
            else:
                print(f"Cannot extract file {data.name} to {shape_extract_folder}")
                return file_type, None

            shapefiles = self.find_shapefiles(shape_extract_folder)
            if not shapefiles:
                for extracted_file in extracted_files:
                    nested_folder = os.path.join(shape_extract_folder, extracted_file)
                    if os.path.isdir(nested_folder):
                        nested_shapefiles = self.find_shapefiles(nested_folder)
                        shapefiles.extend(nested_shapefiles)
            # Use the extraction folder name as the base name for all shape files (sanitized in rename function)
            base_name = os.path.basename(shape_extract_folder)
            if shapefiles:
                for shapefile in shapefiles:
                    self.rename_and_move_files(shapefile, shape_extract_folder, base_name)

            return file_type, shape_extract_folder
        return file_type, None

class ArcGISHandler:
    
    def __init__(self):
        # Try to detect the environment and set appropriate project path
        try:
            # First try to get the current project if running in ArcGIS Pro
            self.aprx_file = "CURRENT"
            # Test if CURRENT works
            test_aprx = arcpy.mp.ArcGISProject(self.aprx_file)
            test_aprx = None  # Clean up
        except (OSError, RuntimeError):
            # If CURRENT doesn't work, fall back to project file from .env
            self.aprx_file = os.environ.get('APRX')
            
            # Verify the project file exists
            if not self.aprx_file or not os.path.exists(self.aprx_file):
                print(f"⚠️  Warning: Project file not found at {self.aprx_file}")
                print("Please check the 'ArcGISProject' path in your .env file")
                # Set to None to handle gracefully later
                self.aprx_file = None
        
        self.map_name = os.environ.get('AR_MAP_NAME')

    @staticmethod
    def _xp(path: str) -> str:
        # Convert to extended-length path for Windows (helps with long paths)
        if not path:
            return path
        norm = os.path.normpath(path)
        if norm.startswith('\\\\?\\') or norm.startswith('\\\\.\\'):
            return norm
        if os.path.splitdrive(norm)[0]:
            return r"\\\\?\\" + norm
        return norm
    
    def find_shapefiles(self, folder):
        shapefiles = []
        for root, _, files in os.walk(folder):
            for file in files:
                if file.lower().endswith('.shp'):
                    shapefiles.append(os.path.join(root, file))
        return shapefiles

    def _ensure_cpg_utf8(self, shapefile_path):
        base = os.path.splitext(shapefile_path)[0]
        cpg_path = base + ".cpg"
        if not os.path.exists(cpg_path):
            try:
                with open(cpg_path, "w", encoding="ascii") as f:
                    f.write("UTF-8")
            except Exception as e:
                print(f"Warning: could not create CPG at {cpg_path}: {e}")

    def shp_add_to_arcgispro(self, shape_extract_folder, cdg_value):
        if not self.aprx_file:
            print("No ArcGIS Pro project available. Open Pro or set ArcGISProject in your .env.")
            return
        # Set workspace using extended-length path when needed
        arcpy.env.workspace = self._xp(shape_extract_folder)
        aprx = arcpy.mp.ArcGISProject(self.aprx_file)
        map_obj = aprx.listMaps(self.map_name)[0]
        shapefiles = self.find_shapefiles(shape_extract_folder)
        
        for shapefile_path in shapefiles:
            try:
                self.shp_add_CdgActvdd(shapefile_path, cdg_value)
                # Use normal path for map add (ArcGIS usually handles it better)
                map_obj.addDataFromPath(shapefile_path)
                print(f"Shapefile '{shapefile_path}' added to the map '{self.map_name}' in the project '{self.aprx_file}'.")
            except arcpy.ExecuteError as e:
                print(f"An error occurred when adding CdgActvdd field: {e}")
            except RuntimeError as e:
                print(f"Failed to add data. Possible credentials issue: {e}")     

    def shp_add_to_arcgispro_c2(self, shape_extract_folder, cdg_value, c2_cdg_dict):
        if not self.aprx_file:
            print("No ArcGIS Pro project available. Open Pro or set ArcGISProject in your .env.")
            return
        arcpy.env.workspace = self._xp(shape_extract_folder)
        aprx = arcpy.mp.ArcGISProject(self.aprx_file)
        map_obj = aprx.listMaps(self.map_name)[0]
        shapefiles = self.find_shapefiles(shape_extract_folder)
        
        for shapefile_path in shapefiles:
            try:
                self.shp_add_CdgActvdd_c2(shapefile_path, cdg_value, c2_cdg_dict)
                map_obj.addDataFromPath(shapefile_path)
                print(f"Shapefile '{shapefile_path}' added to the map '{self.map_name}' in the project '{self.aprx_file}'.")
            except arcpy.ExecuteError as e:
                print(f"An error occurred when adding CdgActvdd field: {e}")
            except RuntimeError as e:
                print(f"Failed to add data. Possible credentials issue: {e}")

    def shp_add_CdgActvdd(self, shapefile_path, cdg_value):
        new_field_name = "CdgActvdd"
        # Robust existence check (normal and extended)
        xp = self._xp(shapefile_path)
        exists_normal = arcpy.Exists(shapefile_path)
        exists_xp = arcpy.Exists(xp)
        if not (exists_normal or exists_xp):
            # Extra debug to help diagnose
            print(f"os.path.exists: {os.path.exists(shapefile_path)} | shapefile: {shapefile_path}")
            raise arcpy.ExecuteError(f"Dataset not found: {shapefile_path}")
        target = xp if exists_xp and not exists_normal else shapefile_path
        # Only add if missing
        existing = {f.name for f in arcpy.ListFields(target)}
        if new_field_name not in existing:
            arcpy.AddField_management(target, new_field_name, "TEXT", field_length=50)
        # Ensure UTF-8 dbf codepage
        self._ensure_cpg_utf8(target)
        # Use PYTHON3 in ArcGIS Pro
        arcpy.CalculateField_management(target, new_field_name, f"'{cdg_value}'", "PYTHON3")

    def shp_add_CdgActvdd_c2(self, shapefile_path, cdg_value, c2_cdg_dict):
        """
        Add CdgActvdd field to a shapefile based on area values for Component 2.
        """
        xp = self._xp(shapefile_path)
        target = xp if arcpy.Exists(xp) else shapefile_path
        # Find area-related field in the shapefile
        field_names = [field.name for field in arcpy.ListFields(target) if 
                    ("rea" in field.name.lower() or "ha" in field.name.lower() or "hectare" in field.name.lower()) and 
                    field.type in ["Double", "Float", "Integer", "SmallInteger", "Single"]]
        
        if not field_names:
            print(f"No suitable area field found in {shapefile_path}")
            # Add the field with default value if no area field is found
            new_field_name = "CdgActvdd"
            try:
                existing = {f.name for f in arcpy.ListFields(target)}
                if new_field_name not in existing:
                    arcpy.AddField_management(target, new_field_name, "TEXT", field_length=50)
                self._ensure_cpg_utf8(target)
                arcpy.CalculateField_management(target, new_field_name, f"'{cdg_value}'", "PYTHON3")
                print(f"Added default code '{cdg_value}' to {shapefile_path}")
            except arcpy.ExecuteError as e:
                print(f"Error adding field: {e}")
            return
        
        # Get the first field that matches the criteria
        area_field = field_names[0]
        print(f"Using area field: {area_field}")
        
        # Calculate total area in the shapefile
        total_area = 0
        with arcpy.da.SearchCursor(target, [area_field]) as cursor:
            for row in cursor:
                if row[0] is not None:
                    total_area += row[0]
        
        print(f"Total area in shapefile: {total_area:.2f}")
        
        # Find the closest matching code based on area
        closest_code = None
        min_diff = float('inf')
        
        for code, area in c2_cdg_dict.items():
            diff = abs(round(total_area, 2) - round(area, 2))
            if diff < min_diff:
                min_diff = diff
                closest_code = code
        
        # Set a threshold for what's considered a match (e.g., 5% of the total area)
        threshold = max(0.05 * total_area, 0.1)  # At least 0.1 ha difference allowed
        
        # Add a new text field
        new_field_name = "CdgActvdd"
        try:
            existing = {f.name for f in arcpy.ListFields(target)}
            if new_field_name not in existing:
                arcpy.AddField_management(target, new_field_name, "TEXT", field_length=50)
        except arcpy.ExecuteError:
            # Field might already exist
            pass
        self._ensure_cpg_utf8(target)
        
        # If we found a close match, use it; otherwise use the default
        if closest_code and min_diff <= threshold:
            print(f"Found matching code {closest_code} with area {c2_cdg_dict[closest_code]:.2f} ha (difference: {min_diff:.2f} ha)")
            arcpy.CalculateField_management(target, new_field_name, f"'{closest_code}'", "PYTHON3")
        else:
            # If the shape file has only one feature with an area value, try to match based on individual features
            if arcpy.GetCount_management(target).getOutput(0) == "1":
                with arcpy.da.UpdateCursor(target, [area_field, new_field_name]) as cursor:
                    for row in cursor:
                        area_value = row[0]
                        closest_code = None
                        min_diff = float('inf')
                        
                        for code, area in c2_cdg_dict.items():
                            diff = abs(round(area_value, 2) - round(area, 2))
                            if diff < min_diff:
                                min_diff = diff
                                closest_code = code
                        
                        if closest_code and min_diff <= threshold:
                            row[1] = closest_code
                            print(f"Assigned code {closest_code} to feature with area {area_value:.2f} ha")
                        else:
                            row[1] = cdg_value
                            print(f"Using default code {cdg_value} for feature with area {area_value:.2f} ha")
                        cursor.updateRow(row)
            else:
                # Use the default value for all features
                arcpy.CalculateField_management(target, new_field_name, f"'{cdg_value}'", "PYTHON3")
                print(f"No close area match found. Using default code {cdg_value}")
                
                # Additionally, print all the codes and areas from the dictionary for debugging
                print("Available codes and areas:")
                for code, area in c2_cdg_dict.items():
                    print(f"  {code}: {area:.2f} ha")

## 3. Ejecutar: Actualización de código y obtención de adjuntos por nombre de código / Execute: Code Update & Get Attachments by Code Name

**ES:** *NOTA: Si no limita las filas desde la fila de inicio hasta la de fin, el proceso puede tardar mucho tiempo.*

**EN:** *NOTE: If you don't limit rows from begin row to end row, the process may take a long time.*

### 3.1. Componente 1 / Component 1

#### 3.1.1. Configuración de variables para ejecución: fila de inicio, fila de fin / Variables Set to execute: begin_row, end_row

In [ ]:
# Usage with Class - updated 2024-06-27

# Create a Updater object for C1
updater_c1 = Updater()
smartsheet_handler_c1 = SmartsheetHandler(sheet_id_c1)
file_handler_c1 = FileHandler()
arcgis_handler_c1 = ArcGISHandler()

In [ ]:
#make a input for the user to input the begin row number and end row number
begin_row_c1 = int(input("Enter the begin row number C1: "))
end_row_c1 = int(input("Enter the end row number C1: "))

#### 3.1.2. Actualizar código / Update Code
#### 3.1.3. Obtener archivos adjuntos / Get attachments

In [ ]:
begin_row_c1 = 908
end_row_c1 = 917
updater_c1.updateCdgActvdd(sheet_id_c1, begin_row_c1-1, end_row_c1)
updater_c1.update_review_status(sheet_id_c1, begin_row_c1, end_row_c1)
smartsheet_handler_c1.process_attachments(file_handler_c1, arcgis_handler_c1, begin_row_c1-1, end_row_c1)

In [ ]:
begin_row_c1 = 887
end_row_c1 = 902
updater_c1.updateCdgActvdd(sheet_id_c1, begin_row_c1-1, end_row_c1)
updater_c1.update_review_status(sheet_id_c1, begin_row_c1, end_row_c1)
smartsheet_handler_c1.process_attachments(file_handler_c1, arcgis_handler_c1, begin_row_c1-1, end_row_c1)

In [ ]:
begin_row_c1 = 907
end_row_c1 = 919
updater_c1.updateCdgActvdd(sheet_id_c1, begin_row_c1-1, end_row_c1)
updater_c1.update_review_status(sheet_id_c1, begin_row_c1, end_row_c1)
smartsheet_handler_c1.process_attachments(file_handler_c1, arcgis_handler_c1, begin_row_c1-1, end_row_c1)

In [ ]:
begin_row_c1 = 919
end_row_c1 = 936

updater_c1.updateCdgActvdd(sheet_id_c1, begin_row_c1-1, end_row_c1)
updater_c1.update_review_status(sheet_id_c1, begin_row_c1, end_row_c1)
smartsheet_handler_c1.process_attachments(file_handler_c1, arcgis_handler_c1, begin_row_c1-1, end_row_c1)

### 3.2. Componente 2 / Component 2

#### 3.2.1. Configuración de variables para ejecución: fila de inicio, fila de fin / Variables Set to execute: begin_row, end_row

In [ ]:
updater_c2 = Updater()
smartsheet_handler_c2 = SmartsheetHandler(sheet_id_c2)
file_handler_c2 = FileHandler()
arcgis_handler_c2 = ArcGISHandler()

In [ ]:
#make a input for the user to input the begin row number and end row number
begin_row_c2 = int(input("Enter the begin row number for C2: "))
end_row_c2 = int(input("Enter the end row number for C2: "))


#### 3.2.2. Actualizar código y revisar estados / Update Code & Review Statuses
#### 3.2.3. Obtener archivos adjuntos / Get attachments

In [ ]:
updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#1
start = 3651
end = 3661
updater_c2.updateCdgActvdd(sheet_id_c2, start, end)
updater_c2.update_review_status(sheet_id_c2, start, end)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, start-1, end)

In [ ]:
#2
begin_row_c2 = 3744
end_row_c2 = 3762

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#3
begin_row_c2 = 2169
end_row_c2 = 2180

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#4
begin_row_c2 = 7090
end_row_c2 = 7109

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#5
begin_row_c2 = 1948
end_row_c2 = 1986

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#6 AR-PPD-025 2024-Q4
begin_row_c2 = 2403
end_row_c2 = 2403

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#7 AR-PMD-008 2025-Q1
begin_row_c2 = 5593
end_row_c2 = 5622

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#8 AR-PMD-004 2024-Q4 No Area
begin_row_c2 = 4747
end_row_c2 = 4760

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#9 AR-PMD-004 2024-Q3 Area
begin_row_c2 = 4736
end_row_c2 = 4746

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#Area_17_20 : AR-PMD-001 2024-Q3
begin_row_c2 = 3993
end_row_c2 = 3999

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#Area_21_22 : AR-PMD-010 2025-Q3
begin_row_c2 = 6537
end_row_c2 = 6548

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#Area_23 : AR-PMD-010 2025-Q2
begin_row_c2 = 6521
end_row_c2 = 6535

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#No Area_1
begin_row_c2 = 4147
end_row_c2 = 4158

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#No Area_2
begin_row_c2 = 3406
end_row_c2 = 3426

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

In [ ]:
#Area_24 : AR-PPD-012 2025-Q1
begin_row_c2 = 2934
end_row_c2 = 2959

updater_c2.updateCdgActvdd(sheet_id_c2, begin_row_c2, end_row_c2)
updater_c2.update_review_status(sheet_id_c2, begin_row_c2, end_row_c2)
smartsheet_handler_c2.process_attachments(file_handler_c2, arcgis_handler_c2, begin_row_c2-1, end_row_c2)

### 3.3. Componente 3 / Component 3

#### 3.3.1. Configuración de variables para ejecución: fila de inicio, fila de fin / Variables Set to execute: begin_row, end_row

In [ ]:
# Maintain a set of all the existing cell values and check if the new values is already in the set
# need to execute this funtion before executing 'updateCdgActvdd' funtion
# existing_cdg_values = set()

# before execute this function, make sure that you refreshed all code lines from the very begin of code

# setting begin row number not to change existing confirmed CdgActvdd
begin_row_c3 = 621

#### 3.3.2. Actualizar código / Update Code

In [ ]:
updater_c3 = Updater()
updater_c3.updateCdgActvdd(sheet_id_c3, begin_row_c3)

#### 3.3.3. Obtener archivos adjuntos / Get attachments

In [ ]:
smartsheet_handler_c3 = SmartsheetHandler(sheet_id_c3)
file_handler_c3 = FileHandler()
arcgis_handler_c3 = ArcGISHandler()

smartsheet_handler_c3.process_attachments(file_handler_c3, arcgis_handler_c3, begin_row_c3)

### Fin de la ejecución / End of Execution

### 2.2.0. Versión original: Obtener archivos adjuntos — NO EJECUTAR / Original Version: Get Attachments — DO NOT EXECUTE

In [ ]:
#Orginal

def get_ssheet_AR(sheet_id):

    # components assign
    if sheet_id == sheet_id_c1:
        compo = "C1"
    elif sheet_id == sheet_id_c2:
        compo = "C2"
    elif sheet_id == sheet_id_c3:
        compo = "C3"
    else:
        print("Smartsheet 'id' doesn't match as of the project AR")
        return None
    #Smartsheet setting with sheet id
    ssheet = smartsheet_client.Sheets.get_sheet(sheet_id)
    return ssheet

def get_cdg_value(sheet_id, row_number):
    ssheet = get_ssheet_AR(sheet_id)
    rows = ssheet.rows
    column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")

    for row in rows[row_number-1:row_number]:
        if row.get_column(column_CdgActvdd.id_).value:
            cdg_value = row.get_column(column_CdgActvdd.id_).value
        else:
            print("row: {}, No hay codigo".format(row.row_number))
            return
    return cdg_value

# Need to update like using date first then assign cdg to that date and then if they found exact or similar area -> assign `cdg`
def get_cdg_area(sheet_id, row_start, row_end):
    row_start = row_start
    ssheet = get_ssheet_AR(sheet_id)
    rows = ssheet.rows

    cdg_value = []
    area_value = []
    
    column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")
    column_HA = ssheet.get_column_by_title("TOTAL DE HECTÁREAS")
    
    for row in rows[row_start:row_end]:
    
        if row.get_column(column_CdgActvdd.id_).value != None and row.get_column(column_HA.id_).value != None:
            cdg_value.append(row.get_column(column_CdgActvdd.id_).value)
            area_value.append(row.get_column(column_HA.id_).value)
            #print(cdg_value, area_value)
        else:
            print("row: {}, No hay codigo".format(row.row_number))
            pass
            
    #print(cdg_value, area_value)        
    return cdg_value, area_value


def create_c2_cdg_area_dict(sheet_id, row_start, row_end):
    cdg_values, area_values = get_cdg_area(sheet_id, row_start, row_end)
    c2_cdg_area_dict = {}

    for cdg, area in zip(cdg_values, area_values):
        c2_cdg_area_dict[cdg] = round(area, 2)

    return c2_cdg_area_dict



def get_org_f_name(sheet_id, row_number):
    row1 = row_number-1
    ssheet = get_ssheet_AR(sheet_id)
    rows = ssheet.rows
    column_org_code = ssheet.get_column_by_title("NÚMERO DE CONTRATO")
    column_org_name = ssheet.get_column_by_title("ORGANIZACIÓN")
    column_org_trimes = ssheet.get_column_by_title("TRIMESTRE QUE REPORTA")

    for row in rows[row1:row_number]:
        
        if row.get_column(column_org_code.id_).value and row.get_column(column_org_trimes.id_).value:
            org_cdg = row.get_column(column_org_code.id_).value
            org_name = row.get_column(column_org_name.id_).value
            org_trimes = row.get_column(column_org_trimes.id_).value
            org_f_name = "{}_{}_{}".format(org_cdg,org_name,org_trimes)
        else:
            print("row: {}, No hay datos de Org or de Trimestre".format(row.row_number))
            return
    return org_f_name

def create_directory(path, dir_name):
    """Create a directory with the given name in the given path"""
    dir_path = os.path.join(path, dir_name)
    if not os.path.exists(dir_path):
        os.mkdir(dir_path)

def extract_zip(file_path, destination_path):
    extracted_files = []
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(destination_path)
        extracted_files = zip_ref.namelist()
    return extracted_files


def extract_rar(file_path, destination_path):
    extracted_files = []
    with rarfile.RarFile(file_path) as rar_ref:
        rar_ref.extractall(destination_path)
        extracted_files = rar_ref.namelist()
    return extracted_files

def find_shapefiles(folder):
    shapefiles = []
    for root, _, files in os.walk(folder):
        for file in files:
            if file.lower().endswith('.shp'):
                shapefiles.append(os.path.join(root, file))
    return shapefiles

def rename_and_move_files(shapefile, shape_extract_folder):
    base_file_name = os.path.splitext(os.path.basename(shapefile))[0]
    for ext in ['.shp', '.dbf', '.prj', '.shx', '.cpg', '.qmd']:
        related_file = os.path.splitext(shapefile)[0] + ext
        if os.path.exists(related_file):
            new_file_name = os.path.join(shape_extract_folder, base_file_name + ext)
            i = 1
            while os.path.exists(new_file_name):
                new_file_name = os.path.join(shape_extract_folder, f"{base_file_name}_{i}{ext}")
                i += 1
            shutil.move(related_file, new_file_name)
            print(f"Moved and renamed file {related_file} to {new_file_name}")

def save_attachment(sheet_id, attachment, cdg_value):
    """Download the attachment and save it to the corresponding directory"""
    data = smartsheet_client.Attachments.get_attachment(sheet_id, attachment.id_)
    print(f"Checking attachment {data.name}")

    # Skip attachments with None URLs
    if data.url is None:
        print(f"Skipping attachment {data.name} because URL is None")
        return

    f_name = os.path.splitext(data.name)[0].lower()
    f_exten_name = os.path.splitext(data.name)[1].lower()
    
    # Check if the attachment's extentions and assign as corresponding, for example, 'xlsx' is beneficiary list, 'zip' is shape file or anexos.
    if f_exten_name in ['.xlsm']:
        extension = f_exten_name
        file_type = 'Reporte-pp'
    elif f_exten_name in ['.xls', '.xlsx']:
        extension = f_exten_name
        # Split f_name to get base name without extension
        #f_base_name = os.path.splitext(f_name)[0]
        #print(f_base_name)
        if any(keyword in f_name for keyword in ['bd', 'bene', 'm&e', 'basedatos']):
            file_type = 'Reporte-pp'
        elif any(keyword in f_name for keyword in ['Área', 'área', 'area', 'hectareas', 'ha', 'SAF', 'saf']):
            file_type = 'Area_ha'
        else:
            file_type = 'Referencia'  
    elif f_exten_name in ['.doc', '.docx', '.pdf', '.pptx', '.ppt']:
        extension = f_exten_name
        file_type = 'Verficador'   
    elif f_exten_name in ['.jpg', '.jpeg', '.png', '.jfif']:
        extension = f_exten_name
        file_type = 'Fotos'
    elif f_exten_name in ['.zip', '.7z', '.rar']: #'.rar' file skip modification
        extension = f_exten_name
        #print(f_base_name)
        if any(keyword in f_name for keyword in ['shape', 'shapes', 'shp', 'ha', 'saf', 'prote', 'plant', 'resta', 'produ']):
            file_type = 'Shapes'
        else:
            file_type = 'Anexos'        
    else:
        extension = ''
        file_type = ''

    # folder path assign depends on components
    if sheet_id == sheet_id_c1:
        path_to_folder = path_to_folder1
    elif sheet_id == sheet_id_c2:
        path_to_folder = path_to_folder2
    elif sheet_id == sheet_id_c3:
        path_to_folder = path_to_folder3
    else:
        print("Smartsheet 'id' doesn't match as of the project AR")
        return
    
    dir_name = str(cdg_value)
    create_directory(path_to_folder, dir_name)
    
    # Use \\?\ prefix to handle long paths
    abs_path_to_folder = os.path.abspath(path_to_folder)

    # Download the attachment to a local file
    
    file_path = os.path.join(abs_path_to_folder, dir_name, f"{cdg_value}-{file_type}{extension}")

    # Check if a file with the same name already exists and if it is true, add number +1
    if os.path.isfile(file_path):
        i = 1
        while os.path.isfile(file_path):
            i += 1
            file_path = os.path.join(path_to_folder, dir_name, f"{cdg_value}-{file_type}-{i}{extension}")

    if 'Anexos' in file_path:
        return file_path, None
    
    attachment_data = requests.get(data.url).content
    
    with open(file_path, 'wb') as f:
        f.write(attachment_data)

    print(f"Downloaded attachment {data.name} to {file_path}")

    # if shapes file, automatically extract 'zip' file so ArcGIS pro can open effectively
    
        
    if 'Shapes' in file_path:
        shape_zip_path = file_path
        shape_extract_folder = os.path.splitext(file_path)[0]
        if extension == '.zip':
            extracted_files = extract_zip(shape_zip_path, shape_extract_folder) 
            print(f"Extracted shapes zip file {data.name} to {file_path}")
        elif extension == '.rar':
            extracted_files = extract_rar(shape_zip_path, shape_extract_folder) 
            print(f"Extracted shapes zip file {data.name} to {file_path}")
        else:
            print(f"Can not extract zip or rar file {data.name} to {file_path}")
            pass
        
        shapefiles = find_shapefiles(shape_extract_folder)
        if not shapefiles:
            for extracted_file in extracted_files:
                nested_folder = os.path.join(shape_extract_folder, extracted_file)
                if os.path.isdir(nested_folder):
                    nested_shapefiles = find_shapefiles(nested_folder)
                    shapefiles.extend(nested_shapefiles)
        
        if shapefiles:
            for shapefile in shapefiles:
                rename_and_move_files(shapefile, shape_extract_folder)
                
        # Loop through the extracted files and rename them
        for extracted_file in extracted_files:

            extracted_file_path = os.path.join(shape_extract_folder, extracted_file)
            #desc = None  # Initialize desc outside the if condition
            
            #if os.path.splitext(extracted_file)[1] == ".shp":
            #    desc = arcpy.Describe(extracted_file_path)
            #    print(arcpy.Describe(extracted_file_path).shapeType)
            
            if 'Poligonos_agrupados' in extracted_file:
                base_file_name = f"new_s123_polygon_{cdg_value}"
            elif 'Puntos_agrupados' in extracted_file:
                base_file_name = f"new_s123_point_{cdg_value}"
            elif 'AR_Monitoreo_Restauracion_v1_0' in extracted_file:
                base_file_name = f"new_s123_general_{cdg_value}"
            elif 'saf' in extracted_file:
                base_file_name = f"new_ssheet_saf_{cdg_value}"
            elif 'prote' in extracted_file:
                base_file_name = f"new_ssheet_prote_{cdg_value}"
            elif 'plant' in extracted_file:
                base_file_name = f"new_ssheet_plant_{cdg_value}"
            elif 'resta' in extracted_file:
                base_file_name = f"new_ssheet_resta_{cdg_value}"
            elif 'produ' in extracted_file:
                base_file_name = f"new_ssheet_produ_{cdg_value}"
            else:
                base_file_name = f"new_ssheet_{cdg_value}"

            new_file_name = os.path.join(shape_extract_folder, f"{base_file_name}{os.path.splitext(extracted_file)[1]}")

            # Check if the new file name already exists, if yes, add numbers to the file name
            i = 1
            while os.path.isfile(new_file_name):
                new_file_name = os.path.join(shape_extract_folder, f"{base_file_name}_{i}{os.path.splitext(extracted_file)[1]}")
                i += 1

            os.rename(extracted_file_path, new_file_name)
            print(f"Renamed file {extracted_file} to {os.path.basename(new_file_name)}")
    
            
        return file_type, shape_extract_folder
    return file_type, None

def shp_add_to_arcgispro(shape_extract_folder, cdg_value):

    # Set workspace
    arcpy.env.workspace = shape_extract_folder

    # Load the project
    aprx_file = os.environ.get('APRX')
    aprx = arcpy.mp.ArcGISProject(aprx_file)

    # Specify the map within the project
    map_name = os.environ.get('AR_MAP_NAME')
    map_obj = aprx.listMaps(map_name)[0]

    # Get a list of all files in the folder
    all_files = os.listdir(shape_extract_folder)

    # Filter files with ".shp" extension
    shapefiles = [file for file in all_files if file.lower().endswith(".shp")]

    # Print the list of shapefiles
    for shapefile in shapefiles:
        shapefile_path = os.path.join(shape_extract_folder, shapefile)

        try:
            # Call shp_add_CdgActvdd with map_obj, new_shapefile_path, and cdg_value
            shp_add_CdgActvdd(shapefile_path, cdg_value)
            
        except arcpy.ExecuteError as e:
            print(f"An error occurred when adding CdgActvdd field: {e}")
            # Handle the "loc" issue here, if needed

        # Add the copied shapefile to the map
        feature_layer = map_obj.addDataFromPath(shapefile_path)
        
        print(f"Shapefile '{shapefile}' added to the map '{map_name}' in the project '{aprx_file}'.")

        # Remove the feature layer from the map (without deleting the shapefile)
        #map_obj.removeLayer(feature_layer)

    # Save the changes to the project
    aprx.save()
    print("Project saved successfully.")

def shp_add_to_arcgispro_c2(shape_extract_folder, cdg_value, c2_cdg_dict):
    # Set workspace
    arcpy.env.workspace = shape_extract_folder

    # Load the project
    aprx_file = "CURRENT"
    aprx = arcpy.mp.ArcGISProject(aprx_file)

    # Specify the map within the project
    map_name = os.environ.get('AR_MAP_NAME')
    map_obj = aprx.listMaps(map_name)[0]

    # Get a list of all files in the folder
    all_files = os.listdir(shape_extract_folder)

    # Filter files with ".shp" extension
    shapefiles = [file for file in all_files if file.lower().endswith(".shp")]

    # Print the list of shapefiles
    for shapefile in shapefiles:
        shapefile_path = os.path.join(shape_extract_folder, shapefile)

        try:
            # Call shp_add_CdgActvdd_c2 with map_obj, new_shapefile_path, cdg_value, c2_cdg, and c2_area
            shp_add_CdgActvdd_c2(shapefile_path, cdg_value, c2_cdg_dict)
            
        except arcpy.ExecuteError as e:
            print(f"An error occurred when adding CdgActvdd field: {e}")
            # Handle the "loc" issue here, if needed

        # Add the copied shapefile to the map
        feature_layer = map_obj.addDataFromPath(shapefile_path)
        
        print(f"Shapefile '{shapefile}' added to the map '{map_name}' in the project '{aprx_file}'.")

    # Save the changes to the project
    aprx.save()
    print("Project saved successfully.")
    
    
        
def shp_add_CdgActvdd_c2(shapefile_path, cdg_value, c2_cdg_dict):
    # Search for a field that contains "area" or "ha" and is of type Double or Float
    field_names = [field.name for field in arcpy.ListFields(shapefile_path) if 
                   ("rea" in field.name.lower() or "ha" in field.name.lower()) and 
                   field.type in ["Double", "Float"]]

    if field_names:
        # Get the first field that matches the criteria
        area_field = field_names[0]

        # Add a new text field
        new_field_name = "CdgActvdd"
        arcpy.AddField_management(shapefile_path, new_field_name, "TEXT")

        # Get the area value from the shapefile
        with arcpy.da.UpdateCursor(shapefile_path, [area_field, new_field_name]) as cursor:
            for row in cursor:
                area_value = row[0]
                # Check if the area value matches any of the values in the c2_cdg_dict
                matching_keys = [key for key, value in c2_cdg_dict.items() if round(value, 2) == round(area_value, 2)]
                if matching_keys:
                    row[1] = matching_keys[0]  # Assign the first matching key
                else:
                    row[1] = ""
                cursor.updateRow(row)
                
        print(f"Calculated 'CdgActvdd' field in {shapefile_path}")
        
    else:
        print(f"No suitable area field found in {shapefile_path}")




def shp_add_CdgActvdd(shapefile_path, cdg_value):
    # Add a new text field and calculate with cdg_value
    new_field_name = "CdgActvdd"
    arcpy.AddField_management(shapefile_path, new_field_name, "TEXT")
    arcpy.CalculateField_management(shapefile_path, new_field_name, f"'{cdg_value}'", "PYTHON")
    

def process_attachments(sheet_id, begin_rowNum, EndRowNum=2022):
    # components assign
    if sheet_id == sheet_id_c1:
        compo = "C1"
    elif sheet_id == sheet_id_c2:
        compo = "C2"
    elif sheet_id == sheet_id_c3:
        compo = "C3"
    else:
        print("Smartsheet 'id' doesn't match as of the project AR")
        return
    
    """Loop through each row and attachment to download and save the attachments"""
    ssheet = get_ssheet_AR(sheet_id)
    rows = ssheet.rows
    row_count = len(ssheet.rows)
    if EndRowNum == 2022:
        EndRowNum = row_count
    else:
        EndRowNum = EndRowNum
        
    if compo == 'C2':
        c2_cdg_area_dict = create_c2_cdg_area_dict(sheet_id, begin_rowNum, EndRowNum)


    for actv in rows[begin_rowNum:EndRowNum]:

        # Include discussions, attachments, columns, and columnType
        row = smartsheet_client.Sheets.get_row(
            sheet_id,       # sheet_id
            actv.id_,       # row_id
            include='discussions,attachments,columns,columnType'
        )

        print(f"Checking row {row.row_number}")

        # Loop through each attachment in the row
        for attachment in row.attachments:

            # IF Smartsheet_ID is component2, get 'orgnization name+quarter' as folder name, #else get code value as folder name
            if compo == 'C2':
                cdg_value = get_org_f_name(sheet_id, row.row_number)

            else: 
                cdg_value = get_cdg_value(sheet_id, row.row_number) 

            if cdg_value != None:
                # Save the attachment to the corresponding directory
                file_type, shape_extract_folder = save_attachment(sheet_id, attachment, cdg_value)
                
                if file_type == 'Shapes' and compo == "C1":
                    # Call shp_add_to_arcgispro and get map_obj and shapefile_path
                    shp_add_to_arcgispro(shape_extract_folder, cdg_value)
                    
                elif file_type == 'Shapes' and compo == "C2":
                    shp_add_to_arcgispro_c2(shape_extract_folder, cdg_value, c2_cdg_area_dict)


### 2.1.0. Versión original: Obtener código — NO EJECUTAR / Original Version: Get Code — DO NOT EXECUTE

In [ ]:
# Maintain a set of all the existing cell values and check if the new values is already in the set
existing_cdg_values = set()

def existing_cdgs(sheet_id):
  #cdg colums values list

    #Smartsheet setting with sheet id
  ssheet = smartsheet_client.Sheets.get_sheet(sheet_id)
  rows = ssheet.rows
  row_count = len(ssheet.rows)
  column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")

  for row in rows[0:row_count]:

    # row attribute using get cell data from column id
    codigo_cell = row.get_column(column_CdgActvdd.id_)
    existing_cdg_values.add(codigo_cell.value)

  return existing_cdg_values


# Generate Cdg in date_component(number)_(if area or pp as text)_alphabetic order
def abc_order(date_srt, text, existing_cdg_list):
    # Generate alphabetic order in same day
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    cdg = f"{date_srt}_{text}_"
    for i, char in enumerate(alphabet):
        new_cdg = cdg + char
        if new_cdg not in existing_cdg_list:
            existing_cdg_list.add(new_cdg)
            return new_cdg
        
    return None


# update Cdg by Sheet_id
def updateCdgActvdd(sheet_id, begin_rowNum, EndRowNum=2022):

  # components assign
  if sheet_id == sheet_id_c1:
    compo = "C1"
  elif sheet_id == sheet_id_c2:
    compo = "C2"
  elif sheet_id == sheet_id_c3:
    compo = "C3"
  else:
    print("Smartsheet id doesn't match as the project")
    pass
    return

  #Smartsheet setting with sheet id
  ssheet = smartsheet_client.Sheets.get_sheet(sheet_id)
  rows = ssheet.rows
  row_count = len(ssheet.rows)
  if EndRowNum == 2022:
    EndRowNum = row_count
  else:
    EndRowNum = EndRowNum

  # Get columnn by title 
  column_FECHA = ssheet.get_column_by_title("FECHA DE LA ACTIVIDAD")
  column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")
  column_QUIEN = ssheet.get_column_by_title("NOMBRE DE QUIEN REPORTA")
  #column_HA = ssheet.get_column_by_title("TOTAL DE HECTÁREAS")
  #column_PP = ssheet.get_column_by_title("TOTAL DE PERSONAS")
  #column_ABE = ssheet.get_column_by_title("ACCIONES DE RESTAURACIÓN AbE")
  #column_TIPOACT = ssheet.get_column_by_title("TIPO DE ACTIVIDAD")
  column_cal_dato = ssheet.get_column_by_title("CALIDAD DE DATOS")
  column_cal_sig = ssheet.get_column_by_title("Calidad SIG")

  if compo == "C2":
    column_ORG = ssheet.get_column_by_title("ORGANIZACIÓN")



  for row in rows[begin_rowNum:EndRowNum]:

    if row.modified_at:

      #check if calidad de datos y calidad sig are confirmed. if confirmed, no need to update "codigo de la actividad"
      val_dato = row.get_column(column_cal_dato.id_).value

      #check if Component 3 so no need to ask about "Calidad SIG"
      if compo == "C3":
        val_sig = "Null"
      else:
        val_sig = row.get_column(column_cal_sig.id_).value

      #Check if already reviewed or not
      #if val_dato == "Rojo" or val_dato == "Amarillo" or val_dato == "Verde" or val_dato == "Sí" or val_dato == "Espera" or val_dato == "No" or val_sig == "Sí" or val_sig == "Espera" or val_sig == "No":
      review = ["Rojo", "Amarillo", "Verde", "Sí", "Espera", "No"]
      if val_dato in review or val_sig in review:
        print("Row number:", row.row_number, val_dato, val_sig)
        pass
      else: 
        print("Row number:", row.row_number, val_dato, val_sig)

        # row attribute using get cell data from column id
        cdg_cell = row.get_column(column_CdgActvdd.id_)
        fecha_cell = row.get_column(column_FECHA.id_)
        #ha_cell = row.get_column(column_HA.id_)
        #pp_cell = row.get_column(column_PP.id_)
        quien_cell = row.get_column(column_QUIEN.id_)

        if compo == "C2":
          org_cell = row.get_column(column_ORG.id_)


        # If "fecha" doesnt assgined, just pass becaues it isn't possible generating "codigo de la actividad"
        if fecha_cell.value == None:
          print("No fechas")
          pass

        # if only "fecha" exist
        else:
          fecha = fecha_cell.value.replace("-","")

          #print(fecha_cell.value, cdg_cell.value, quien_cell.value, ha_cell.value, pp_cell.value)

          # Generate CdgActvdd depends on Abe, area of Ha, number of people
          # date + component + 3 letter of reporter + AbE + hectares + number of people + smartsheet Row number
          # but I can use "area or pp as string to identificate data attribute"
          
          #conca_values = abc_order(str(fecha[2:]), text, existing_cdg_values)
          if compo == "C2":
            dequien = org_cell.value
          else:
            dequien = str(quien_cell.value[:3])

          if dequien != None:
            text = compo + "_"+ dequien
          else:
            print("There is no name of Org")
            return

          # create code
          conca_values = abc_order(str(fecha[2:]), text, existing_cdgs(sheet_id))

          updated_cell = smartsheet.models.Cell({
              "column_id": column_CdgActvdd.id_,
              "value": conca_values
          })
          updated_row = smartsheet.models.Row({
              "id": row.id,
              "cells": [updated_cell]
          })
          
          try:
              # Update the rows
              result = smartsheet_client.Sheets.update_rows(sheet_id, [updated_row]).data
              # Print the updated cell value
              print(updated_row.cells[0].value)

          except smartsheet.exceptions.SmartsheetAPIError as e:
              print("An error occured: %s" % e)

    else:
      print("End of Row \n")



def update_colums(sheet_id, column_name, new_cell_value, begin_rowNum, EndRowNum=2022):

  #Smartsheet setting with sheet id
  ssheet = smartsheet_client.Sheets.get_sheet(sheet_id)
  rows = ssheet.rows
  row_count = len(ssheet.rows)
  if EndRowNum == 2022:
    EndRowNum = row_count
  else:
    EndRowNum = EndRowNum

  # Get columnn by title 
  column_name = ssheet.get_column_by_title(column_name)
  column_CdgActvdd = ssheet.get_column_by_title("CÓDIGO DE LA ACTIVIDAD")
  

  for row in rows[begin_rowNum:EndRowNum]:

    if row.modified_at:

        # row attribute using get cell data from column id
        codigo_cell = row.get_column(column_CdgActvdd.id_)

        # If "fecha" doesnt assgined, just pass becaues it isn't possible generating "codigo de la actividad"
        if codigo_cell.value == None:
          print("No Codigo")
          pass

        # if only "Codigo" exist
        else:

          updated_cell = smartsheet.models.Cell({
              "column_id": column_name.id_,
              "value": new_cell_value
          })
          updated_row = smartsheet.models.Row({
              "id": row.id,
              "cells": [updated_cell]
          })
          
          try:
              # Update the rows
              result = smartsheet_client.Sheets.update_rows(sheet_id, [updated_row]).data
              # Print the updated cell value
              print(updated_row.cells[0].value)

          except smartsheet.exceptions.SmartsheetAPIError as e:
              print("An error occured: %s" % e)

    else:
      print("End of Row \n")


## 4. Control de columnas de Smartsheet / Smartsheet Column Control

### 4.1. C1: Mostrar u ocultar columnas / C1: Show or hide columns

In [ ]:
# Get all the columns for the sheet
columns_result = smartsheet_client.Sheets.get_columns(sheet_id_c1)
columns = columns_result.data


for column in columns:
    
    print(f'{column.title}: {column.id_}')

In [ ]:
# Ss_C1_Visible
column_spec = smartsheet.models.Column({
    'title': 'OFICIAL',
    'hidden': False
})


updated_column = smartsheet_client.Sheets.update_column(sheet_id_c1, 0  # TODO: set your column_id (was a real Smartsheet column id), column_spec)
#print(f"Column {updated_column.title} updated successfully.")

In [ ]:
# update_oficial data for component 1

begin_row_oficial = 390
end_row_oficial = 405
Datos_trimestral_oficial = '2023-Q2'

update_colums(sheet_id_c1,'OFICIAL', Datos_trimestral_oficial, begin_row_oficial, end_row_oficial)


In [ ]:
# Ss_C1_Hide
column_spec = smartsheet.models.Column({
    'title': 'OFICIAL',
    'hidden': True
})


updated_column = smartsheet_client.Sheets.update_column(sheet_id_c1, 0  # TODO: set your column_id (was a real Smartsheet column id), column_spec)
#print(f"Column {updated_column.title} updated successfully.")

In [ ]:
# Ss_C1_Codigo culumn -> lock
column_spec = smartsheet.models.Column({
    #'title': 'CÓDIGO DE LA ACTIVIDAD',
    'locked': True
})

# column_ID: column that you want to make a lock
updated_column = smartsheet_client.Sheets.update_column(sheet_id_c1, 0  # TODO: set your column_id (was a real Smartsheet column id), column_spec)
#print(f"Column {updated_column.title} updated successfully.")

### 4.2. C2: Mostrar u ocultar columnas / C2: Show or hide columns

In [ ]:
# Get all the columns for the sheet for C2
columns_result = smartsheet_client.Sheets.get_columns(sheet_id_c2)
columns = columns_result.data


for column in columns:
    
    print(f'{column.title}: {column.id_}')

In [ ]:
# Ss_C2_Visible - Show
column_spec = smartsheet.models.Column({
    'title': 'MONTO DEL INCENTIVO (QQ)',
    'hidden': False
})

updated_column = smartsheet_client.Sheets.update_column(
    sheet_id_c2, 
    0  # TODO: set your column_id (was a real Smartsheet column id), 
    column_spec)

In [ ]:
# Ss_C2_Visible - Show
column_spec = smartsheet.models.Column({
    'title': 'OFICIAL',
    'hidden': False
})

updated_column = smartsheet_client.Sheets.update_column(
    sheet_id_c2, 
    0  # TODO: set your column_id (was a real Smartsheet column id), 
    column_spec)

In [ ]:
# update_oficial data for component 2

begin_row_oficial = 390
end_row_oficial = 405
Datos_trimestral_oficial = '2023-Q2'

#update_colums(sheet_id_c2,'OFICIAL', Datos_trimestral_oficial, begin_row_oficial, end_row_oficial)


In [ ]:
# Ss_C2_Visible - Hide
column_spec = smartsheet.models.Column({
    'title': 'OFICIAL',
    'hidden': True
})

updated_column = smartsheet_client.Sheets.update_column(
    sheet_id_c2, 
    0  # TODO: set your column_id (was a real Smartsheet column id), #column_id
    column_spec)

In [ ]:
# Ss_C2_lock - True for the column 'SIG'
column_spec = smartsheet.models.Column({
    #'title': 'Calidad SIG',
    'locked': True
})

updated_column = smartsheet_client.Sheets.update_column(
    sheet_id_c2, 
    5585966628398980, #column_id
    column_spec)

### 4.3. C3: Mostrar u ocultar columnas / C3: Show or hide columns

In [ ]:
# Get all the columns for the sheet for C3

columns_result = smartsheet_client.Sheets.get_columns(sheet_id_c3)
columns = columns_result.data


for column in columns:
    
    print(f'{column.title}: {column.id_}')

In [ ]:
# Ss_C3_Visible
column_spec = smartsheet.models.Column({
    'title': 'OFICIAL',
    'hidden': False
})
updated_column = smartsheet_client.Sheets.update_column(
    sheet_id_c3, 
    2686208834332548, 
    column_spec)


In [ ]:
# Ss_C3_Visible - Hide
column_spec = smartsheet.models.Column({
    'title': 'OFICIAL',
    'hidden': True
})
updated_column = smartsheet_client.Sheets.update_column(
    sheet_id_c3, 
    2686208834332548, 
    column_spec)


In [ ]:
# update_oficial data for component 3

begin_row = 337
end_row = 370
#update_colums(sheet_id_c3,'OFICIAL', '2023-Q1', begin_row, end_row)


### 4.4. Crear nuevas columnas al final o eliminar columnas / Create new columns at end or delete columns

In [ ]:
#NO EXCUTE : Make a new column

last_column_index = len(columns)
new_column_index = last_column_index +1

# Set the column properties
column_spec = smartsheet.models.Column({
    'title': 'TimeControl',
    'type': 'TEXT_NUMBER',
    'index': new_column_index,  # set the index to specify the position of the column
})

# Create the column
#new_column = smartsheet_client.Sheets.add_columns(sheet_id_c1, [column_spec]) # title은 유니크해야 하므로 2회 실행시에는 오류 발생

print(f'New column created with ID {columns[-1].id_} and Title {columns[-1].title}')

In [ ]:
print(f'Last column with ID: {columns[-1].id_} and Title: {columns[-1].title}')

title = 'TimeControl'
clmn = next((c for c in columns if c.title == title), None)

if clmn is not None:
    clmn.index = -1
    #updated_column = smartsheet_client.Sheets.update_column(sheet_id_c1,1257433605138308, clmn )
    print("ok")



In [ ]:
# Delete the column
#smartsheet_client.Sheets.delete_column(sheet_id_c1, 1257433605138308)

## 5. Revisión de códigos duplicados / Code Duplicate Review

In [ ]:
import pandas as pd
df_code = pd.read_clipboard()
df_code.head()


In [ ]:
# Add a new 'CÓDIGO' column and set '211209_C3_diplomado' as the value for the first row

# Rename the '211209_C3_diplomado' column to 'CÓDIGO'
df_code.rename(columns={'211209_C3_diplomado': 'CÓDIGO'}, inplace=True)

df_code.head()

In [ ]:
df_code = df_code.shift(1)
df_code.head()

In [ ]:
# Set '211209_C3_diplomado' as the value for the first row in 'CÓDIGO' column
df_code.at[0, 'CÓDIGO'] = '211209_C3_diplomado'
df_code.head()

In [ ]:
duplicated = df_code['CÓDIGO'].duplicated(keep=False)
duplicated_row = df_code[duplicated]
duplicated_row

In [ ]:
# Count the frequency of each unique value in the 'CÓDIGO' column
value_counts = df_code['CÓDIGO'].value_counts()

# Filter out the counts that are greater than 1 to identify duplicates
duplicates = value_counts[value_counts > 1]

if duplicates.any():
    print(f"The duplicated codes and their frequencies are:\n{duplicates}")
else:
    print(f"There are No duplicated codes and their maximun fequencies are:\n{value_counts.max()}")


Fin. / End.

# Mapa de Incentivos AR / AR Incentives Map

In [ ]:
# Minimal functions: C1 incentives filter + add shapefiles to ArcGIS Pro
from typing import Optional, Tuple


def _has_any_incentivo_value_for_row(sheet, row) -> bool:
    """
    Returns True if any of the 2020–2025 | MONTO DEL INCENTIVO (QQ) columns
    has a numeric value in the given row.
    """
    year_cols = []
    for year in range(2020, 2026):
        title = f"{year} | MONTO DEL INCENTIVO (QQ)"
        try:
            col = sheet.get_column_by_title(title)
            year_cols.append(col)
        except Exception:
            # Column might not exist in this sheet; ignore
            continue

    if not year_cols:
        return False

    for col in year_cols:
        try:
            val = row.get_column(col.id_).value
        except Exception:
            val = None
        if isinstance(val, (int, float)):
            return True
        # Accept numeric strings too
        if isinstance(val, str):
            try:
                float(val.replace(",", ""))
                return True
            except Exception:
                pass
    return False


def _is_shape_zip_attachment(att) -> bool:
    name = (att.name or "").lower()
    return name.endswith(".zip") and ("shape" in name or "shp" in name)


def process_shape_attachments_with_incentivo_values_c1(self, file_handler, arcgis_handler, begin_rowNum: int, EndRowNum: Optional[int] = None) -> int:
    """
    Component 1 only: For rows that have any 2020–2025 incentive values,
    process only shapefile (.zip) attachments whose names include shape/shp,
    then add those shapefiles to ArcGIS Pro using arcpy.

    Returns the count of shapefile attachments processed.
    """
    if self.sheet_id != sheet_id_c1:
        print("This function is only for Component 1 sheet.")
        return 0

    ssheet = self.get_ssheet_AR()
    if ssheet is None:
        return 0

    rows = ssheet.rows
    row_count = len(rows)
    if EndRowNum is None or EndRowNum > row_count:
        EndRowNum = row_count

    processed = 0
    for actv in rows[begin_rowNum:EndRowNum]:
        row = smartsheet_client.Sheets.get_row(
            self.sheet_id, actv.id_, include='discussions,attachments,columns,columnType'
        )

        if not _has_any_incentivo_value_for_row(ssheet, row):
            continue

        cdg_value = self.get_cdg_value(row.row_number)
        if not cdg_value:
            continue

        for attachment in row.attachments:
            if not _is_shape_zip_attachment(attachment):
                continue

            file_type, shape_extract_folder = file_handler.save_attachment(self.sheet_id, attachment, cdg_value)
            if file_type == 'Shapes' and shape_extract_folder:
                arcgis_handler.shp_add_to_arcgispro(shape_extract_folder, cdg_value)
                processed += 1
    return processed

# Monkey-patch onto SmartsheetHandler so you can call:
# SmartsheetHandler.process_shape_attachments_with_incentivo_values_c1(...)
SmartsheetHandler.process_shape_attachments_with_incentivo_values_c1 = process_shape_attachments_with_incentivo_values_c1

In [ ]:
# C1-only: Incentive-based shapefile processor + quick runner
from typing import Optional

def _has_any_incentivo_value_for_row(sheet, row) -> bool:
    for year in range(2020, 2026):
        title = f"{year} | MONTO DEL INCENTIVO (QQ)"
        try:
            col = sheet.get_column_by_title(title)
        except Exception:
            continue
        try:
            val = row.get_column(col.id_).value
        except Exception:
            val = None
        if isinstance(val, (int, float)):
            return True
        if isinstance(val, str):
            try:
                float(val.replace(',', ''))
                return True
            except Exception:
                pass
    return False


def _is_shape_zip_attachment(att) -> bool:
    name = (att.name or '').lower()
    return name.endswith('.zip') and ('shape' in name or 'shp' in name)


def process_shape_attachments_with_incentivo_values_c1(self, file_handler, arcgis_handler, begin_rowNum: int, EndRowNum: Optional[int] = None) -> int:
    if self.sheet_id != sheet_id_c1:
        print('This function is only for Component 1 sheet.')
        return 0
    ssheet = self.get_ssheet_AR()
    if ssheet is None:
        return 0
    rows = ssheet.rows
    row_count = len(rows)
    if EndRowNum is None or EndRowNum > row_count:
        EndRowNum = row_count
    processed = 0
    for actv in rows[begin_rowNum:EndRowNum]:
        row = smartsheet_client.Sheets.get_row(
            self.sheet_id, actv.id_, include='discussions,attachments,columns,columnType'
        )
        if not _has_any_incentivo_value_for_row(ssheet, row):
            continue
        cdg_value = self.get_cdg_value(row.row_number)
        if not cdg_value:
            continue
        for attachment in row.attachments:
            if not _is_shape_zip_attachment(attachment):
                continue
            file_type, shape_extract_folder = file_handler.save_attachment(self.sheet_id, attachment, cdg_value)
            if file_type == 'Shapes' and shape_extract_folder:
                arcgis_handler.shp_add_to_arcgispro(shape_extract_folder, cdg_value)
                processed += 1
    return processed

# Attach to SmartsheetHandler
SmartsheetHandler.process_shape_attachments_with_incentivo_values_c1 = process_shape_attachments_with_incentivo_values_c1

# Runner (C1 only) — adjust row range as needed
try:
    PROCESSED_ATTACHMENT_IDS.clear()
except NameError:
    pass

begin_row_c1 = 50
end_row_c1 = 90

try:
    smartsheet_handler_c1
except NameError:
    smartsheet_handler_c1 = SmartsheetHandler(sheet_id_c1)
try:
    file_handler_c1
except NameError:
    file_handler_c1 = FileHandler()
try:
    arcgis_handler_c1
except NameError:
    arcgis_handler_c1 = ArcGISHandler()

count_c1 = smartsheet_handler_c1.process_shape_attachments_with_incentivo_values_c1(
    file_handler_c1, arcgis_handler_c1, begin_row_c1, end_row_c1
)
print(f"Processed {count_c1} C1 shapefile attachment(s) in rows {begin_row_c1}..{end_row_c1}")